# PDCP · N03 — Stimulus generation (PDSG Stage 1 + Stage 2) → PDCP-Synth-v1
**Notebook 3 of 13.** Kaggle: Accelerator **None (CPU)** · Internet **On** · **Save & Run All (Commit)**. Expected 2–3 h (hard cap set by `MAX_HOURS`).

**Input:** N02 output dataset (`base_drawings.parquet`, `mooney_style.parquet`, `design_config.json`, `pdcp_utils.py`).

**Design (fixed from N01/N02 results)**
- Every pair = two drawings of the same graph. **B** = natural drawing at stress level 0.30–0.50. **A** = new drawing whose stress is `B − Δ` and whose three proxies are set **relative to B**:
  `A_p = B_p + c_p · m_p`, margins m = NU 0.05, ELD 0.05, XR 0.04.

| condition | c (NU, ELD, XR) | meaning |
|---|---|---|
| congruent | (−1,−1,−1) | all proxies favour the lower-stress drawing A |
| neutral | (0,0,0) | proxies identical — only stress differs |
| conflict_nu / _eld / _xr | one +1 | that proxy favours B (the WRONG answer) |
| conflict_all | (+1,+1,+1) | every proxy points to the wrong answer |
- Δ ∈ {0, 0.03, 0.06, 0.10, 0.15, 0.20}; at Δ = 0 the correct answer is "same" and the proxies are lures. Same six conditions at every Δ ⇒ proxy differences are independent of Δ by design.
- **Stage 2:** over-generate, then select a balanced set (equal count per family × Δ × condition) and greedy-swap within cells until |corr(Δstress, Δproxy)| ≤ 0.05 for every proxy (Pearson and Spearman).
- **Counterfactuals (E2):** for a stratified 400-pair subset, A′ = A with ONE proxy raised by δ, stress and other proxies fixed (δ from `design_config.json`).
- **E1b:** pairs of Mooney-style drawings (stress-only, 0.40–0.80) to test whether models exploit degenerate geometry.
- **Few-shot pool:** 30 pairs on held-out graphs (never evaluated).

**Outputs** (`/kaggle/working/n03_output/`): `pdcp_utils.py` v0.1.2, `drawings.parquet`, `pairs.parquet`, `candidates.parquet`, `stage2_report.json`, `images/768/*.png`, figures, `MANIFEST.txt`.

In [1]:
# ==== SETTINGS ====
INPUT_DIR = "/kaggle/input/datasets/resadussalehinrafsan/kbs-n2-output/n02_output"
OUTPUT_DIR = "/kaggle/working/n03_output"
SEED = 2026
FAST = False
N_JOBS = 4
MAX_HOURS = 9.0                 # stop launching new generation chunks after this (Kaggle limit is 12 h)

STRESS_LEVELS = [0.30, 0.35, 0.40, 0.45, 0.50]
STRESS_FLOOR = 0.295            # lowest allowed stress for drawing A
DSTRESS = [0.0, 0.03, 0.06, 0.10, 0.15, 0.20]
CONDS = {"congruent": (-1, -1, -1), "neutral": (0, 0, 0), "conflict_nu": (1, 0, 0),
         "conflict_eld": (0, 1, 0), "conflict_xr": (0, 0, 1), "conflict_all": (1, 1, 1)}
MARGIN = {"nu": 0.05, "eld": 0.05, "xr": 0.04}
TOL_A = {"ksm": 0.004, "nu": 0.02, "eld": 0.02, "xr": 0.01}
EVAL_GRAPHS = [0, 1, 2, 3, 4, 5]     # graphs 0-2 reuse N02 base drawings
FEWSHOT_GRAPH = 6
PER_CELL = 6                         # pairs kept per family x dstress x condition
CANDIDATES = {0.0: 10, 0.03: 10, 0.06: 10, 0.10: 10, 0.15: 14, 0.20: 18}
CORR_LIMIT = 0.05
N_SUBSET = 400                       # E2 counterfactual + E3/E4 ablation subset
N_E1B = 300
N_FEWSHOT = 30
RES = 768

In [2]:
import os, sys, json, time, glob, shutil, subprocess, importlib, math, itertools
os.makedirs(OUTPUT_DIR, exist_ok=True)
for p in ["joblib", "pyarrow"]:
    try: importlib.import_module(p)
    except ImportError: subprocess.run([sys.executable, "-m", "pip", "install", "-q", p], check=True)
import numpy as np, pandas as pd, networkx as nx, torch, matplotlib
matplotlib.use("Agg"); import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from scipy.stats import spearmanr, pearsonr

src = open(f"{INPUT_DIR}/pdcp_utils.py").read()
if '__version__ = "0.1.1"' in src:
    src = src.replace('__version__ = "0.1.1"', '__version__ = "0.1.2"')
    src = src.replace('    """Return a connected simple graph with nodes relabelled 0..n-1."""\n    family = family.upper()',
                      '    """Return a connected simple graph with nodes relabelled 0..n-1."""\n    family = str(family).upper(); n = int(n); seed = int(seed)')
    src = src.replace("rng = np.random.default_rng(seed)\n    torch.manual_seed(seed)",
                      "seed = int(seed)\n    rng = np.random.default_rng(seed)\n    torch.manual_seed(seed)")
assert "seed = int(seed)" in src and '"0.1.2"' in src, "patch to v0.1.2 failed"
open(f"{OUTPUT_DIR}/pdcp_utils.py", "w").write(src)
sys.path.insert(0, OUTPUT_DIR)
import pdcp_utils as U; importlib.reload(U)
CFG = json.load(open(f"{INPUT_DIR}/design_config.json"))
CF_DELTAS = CFG["counterfactual_deltas"]
B02 = pd.read_parquet(f"{INPUT_DIR}/base_drawings.parquet")
MO02 = pd.read_parquet(f"{INPUT_DIR}/mooney_style.parquet")
FAMILY_SETS = sorted({(r.family, int(r.n)) for r in B02.itertuples()})
PROX = ["nu", "eld", "xr"]
if FAST:
    FAMILY_SETS = [("ER", 25), ("TREE", 50)]; EVAL_GRAPHS = [0, 1]; PER_CELL = 2
    CANDIDATES = {k: 3 for k in CANDIDATES}; N_SUBSET = 20; N_E1B = 20; N_FEWSHOT = 4
print("utils", U.__version__, "| families", FAMILY_SETS, "| CF deltas", CF_DELTAS)
T0 = time.time()
def elapsed_h(): return (time.time() - T0) / 3600

utils 0.1.2 | families [('BA', 50), ('BA', 100), ('ER', 25), ('ER', 50), ('ER', 100), ('GRID', 49), ('SBM', 50), ('SBM', 100), ('TREE', 50), ('WS', 50)] | CF deltas {'nu': [0.05, 0.1], 'eld': [0.05, 0.1, 0.15], 'xr': [0.02, 0.04, 0.06]}


## 1. Base pool B (natural drawings): reuse N02 for graphs 0–2, generate graphs 3–6

In [3]:
def make_base(fam, n, gi, s):
    import sys; sys.path.insert(0, OUTPUT_DIR)
    import torch, pdcp_utils as U; torch.set_num_threads(1)
    gseed = SEED + 100 * gi
    gc = U.GraphCache(U.make_graph(fam, n, gseed))
    for attempt in range(4):
        r = U.pdsg_optimise(gc, {"ksm": s}, seed=gseed + 7 * attempt + int(s * 1000))
        m = U.all_metrics(r.X, gc, with_realism=True)
        if r.converged and m["realistic"]: break
    return {"family": fam, "n": n, "graph": gi, "graph_seed": gseed, "stress_target": s,
            "ok": bool(r.converged and m["realistic"]), **{k: m[k] for k in ["ksm", "nu", "eld", "xr", "aspect", "nn_max", "nn_min", "dense"]},
            "coords": r.X.ravel().round(6).tolist()}

reuse = B02[B02.converged & B02.realistic & B02.apply(lambda r: (r.family, int(r.n)) in FAMILY_SETS, axis=1)].copy()
reuse["ok"] = True
reuse = reuse[[c for c in reuse.columns if c in ["family", "n", "graph", "graph_seed", "stress_target", "ok", "ksm", "nu", "eld",
                                                  "xr", "aspect", "nn_max", "nn_min", "dense", "coords"]]]
new_graphs = [g for g in EVAL_GRAPHS + [FEWSHOT_GRAPH] if g > 2]
jobs = [(f, n, gi, s) for f, n in FAMILY_SETS for gi in new_graphs for s in STRESS_LEVELS]
newb = pd.DataFrame(Parallel(n_jobs=N_JOBS, verbose=2)(delayed(make_base)(*j) for j in jobs))
POOL = pd.concat([reuse, newb[newb.ok]], ignore_index=True)
POOL["n"] = POOL.n.astype(int); POOL["graph"] = POOL.graph.astype(int); POOL["graph_seed"] = POOL.graph_seed.astype(int)
POOL["drawing_id"] = POOL.apply(lambda r: f"B_{r.family}{r.n}_g{r.graph}_s{int(round(r.stress_target*100))}", axis=1)
print(len(POOL), "base drawings; new ok rate", round(newb.ok.mean(), 3), f"| {elapsed_h()*60:.1f} min")

[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  33 tasks      | elapsed:   20.2s
[Parallel(n_jobs=4)]: Done 154 tasks      | elapsed:  4.6min


339 base drawings; new ok rate 0.98 | 5.3 min


[Parallel(n_jobs=4)]: Done 200 out of 200 | elapsed:  5.3min finished


## 2. Stage 1 — generate drawing A for each candidate pair

In [4]:
def gen_A(job):
    import sys; sys.path.insert(0, OUTPUT_DIR)
    import numpy as np, torch, pdcp_utils as U; torch.set_num_threads(1)
    gc = U.GraphCache(U.make_graph(job["family"], job["n"], job["graph_seed"]))
    c = CONDS[job["cond"]]
    tg = {"ksm": job["B_ksm"] - job["dstress"]}
    for p, ci in zip(PROX, c): tg[p] = job["B_" + p] + ci * MARGIN[p]
    out = {**{k: job[k] for k in ["cand_id", "family", "n", "graph", "graph_seed", "dstress", "cond", "B_id", "role"]},
           **{"target_" + k: v for k, v in tg.items()}}
    if tg["nu"] < 0 or tg["xr"] < 0 or tg["eld"] < 0:
        return {**out, "ok": False, "reason": "target<0", "seconds": 0.0}
    t = time.time()
    r0 = U.pdsg_optimise(gc, {"ksm": tg["ksm"]}, seed=job["seed"], max_steps=1500, max_restarts=1)   # independent geometry
    r = U.pdsg_optimise(gc, tg, X0=r0.X, tol=TOL_A, max_steps=1200, max_restarts=1, seed=job["seed"] + 1)
    m = U.all_metrics(r.X, gc, with_realism=True)
    reason = "ok" if (r.converged and m["realistic"]) else ("no convergence" if not r.converged else "unrealistic")
    return {**out, "ok": bool(r.converged and m["realistic"]), "reason": reason, "seconds": round(time.time() - t, 2),
            **{k: m[k] for k in ["ksm", "nu", "eld", "xr", "aspect", "nn_max", "nn_min", "dense"]},
            "coords": r.X.ravel().round(6).tolist()}

rng = np.random.default_rng(SEED)
def pool_rows(fam, n, g): return POOL[(POOL.family == fam) & (POOL.n == n) & (POOL.graph == g)]
jobs, cid = [], 0
for fam, n in FAMILY_SETS:
    for d in DSTRESS:
        for cond in CONDS:
            k = CANDIDATES[d]
            for i in range(k):
                g = EVAL_GRAPHS[i % len(EVAL_GRAPHS)]
                rows = pool_rows(fam, n, g)
                rows = rows[rows.ksm - d >= STRESS_FLOOR]
                if not len(rows): continue
                b = rows.iloc[int(rng.integers(len(rows)))]
                jobs.append({"cand_id": f"c{cid:05d}", "role": "main", "family": fam, "n": int(n), "graph": int(g),
                             "graph_seed": int(b.graph_seed), "dstress": d, "cond": cond, "B_id": b.drawing_id,
                             **{"B_" + k2: float(b[k2]) for k2 in ["ksm", "nu", "eld", "xr"]}, "seed": int(rng.integers(1e8))})
                cid += 1
    # few-shot candidates on the held-out graph
    rows = pool_rows(fam, n, FEWSHOT_GRAPH)
    for i in range(6 if not FAST else 2):
        if not len(rows): break
        d = [0.0, 0.06, 0.10, 0.15, 0.03, 0.0][i % 6]; cond = list(CONDS)[i % 6]
        rr = rows[rows.ksm - d >= STRESS_FLOOR]
        if not len(rr): continue
        b = rr.iloc[int(rng.integers(len(rr)))]
        jobs.append({"cand_id": f"c{cid:05d}", "role": "fewshot", "family": fam, "n": int(n), "graph": FEWSHOT_GRAPH,
                     "graph_seed": int(b.graph_seed), "dstress": d, "cond": cond, "B_id": b.drawing_id,
                     **{"B_" + k2: float(b[k2]) for k2 in ["ksm", "nu", "eld", "xr"]}, "seed": int(rng.integers(1e8))})
        cid += 1
print(len(jobs), "candidate jobs")

CHUNK = 400; results = []
for s in range(0, len(jobs), CHUNK):
    if elapsed_h() > MAX_HOURS * 0.6:
        print("time budget for Stage 1 reached; continuing with", len(results)); break
    results += Parallel(n_jobs=N_JOBS)(delayed(gen_A)(j) for j in jobs[s:s + CHUNK])
    ok = np.mean([r["ok"] for r in results])
    print(f"{len(results)}/{len(jobs)} done | ok {ok:.2f} | {elapsed_h()*60:.0f} min")
CAND = pd.DataFrame(results)
CAND.drop(columns=["coords"], errors="ignore").to_parquet(f"{OUTPUT_DIR}/candidates.parquet", index=False)
print(CAND.groupby("dstress").ok.mean().round(2).to_dict())
print(CAND.groupby("cond").ok.mean().round(2).to_dict())

3966 candidate jobs
400/3966 done | ok 0.88 | 12 min
800/3966 done | ok 0.85 | 41 min
1200/3966 done | ok 0.84 | 48 min
1600/3966 done | ok 0.82 | 58 min
2000/3966 done | ok 0.83 | 74 min
2400/3966 done | ok 0.82 | 84 min
2800/3966 done | ok 0.81 | 100 min
3200/3966 done | ok 0.80 | 169 min
3600/3966 done | ok 0.81 | 174 min
3966/3966 done | ok 0.80 | 189 min
{0.0: 0.95, 0.03: 0.95, 0.06: 0.98, 0.1: 0.93, 0.15: 0.79, 0.2: 0.28}
{'conflict_all': 0.71, 'conflict_eld': 0.84, 'conflict_nu': 0.77, 'conflict_xr': 0.79, 'congruent': 0.87, 'neutral': 0.85}


## 3. Stage 2 — balanced selection + set-level decorrelation

In [5]:
Bm = POOL.set_index("drawing_id")
OK = CAND[CAND.ok & (CAND.role == "main")].copy()
for p in ["ksm"] + PROX:
    OK["d_" + p] = OK.B_id.map(Bm[p]) - OK[p]          # positive = B has the higher value
OK["cell"] = OK.family + OK.n.astype(str) + "|" + OK.dstress.astype(str) + "|" + OK.cond

def corr_stats(df):
    out = {}
    for p in PROX:
        x, y = df["d_ksm"].values, df["d_" + p].values
        out[p] = {"pearson": float(pearsonr(x, y)[0]), "spearman": float(spearmanr(x, y)[0])}
    return out
def objective(df):
    cs = corr_stats(df); return sum(abs(v["pearson"]) + abs(v["spearman"]) for v in cs.values()), cs

rs = np.random.default_rng(SEED + 1)
sel_idx = []
for cell, g in OK.groupby("cell"):
    # prefer distinct graphs inside a cell
    g = g.sample(frac=1, random_state=int(rs.integers(1e6))).sort_values("graph", kind="stable")
    g = g.assign(_r=g.groupby("graph").cumcount()).sort_values("_r", kind="stable")
    sel_idx += list(g.index[:PER_CELL])
sel = set(sel_idx)
before_obj, before_cs = objective(OK.loc[list(sel)])
pool_by_cell = {c: set(g.index) for c, g in OK.groupby("cell")}
best_obj, cs = before_obj, before_cs
no_imp, it = 0, 0
while no_imp < 500 and it < 20000:
    it += 1
    if all(abs(v["pearson"]) <= CORR_LIMIT and abs(v["spearman"]) <= CORR_LIMIT for v in cs.values()): break
    cur = OK.loc[list(sel)]
    worst = max(PROX, key=lambda p: abs(cs[p]["pearson"]))
    zx = (cur.d_ksm - cur.d_ksm.mean()) / (cur.d_ksm.std() + 1e-12)
    zy = (cur["d_" + worst] - cur["d_" + worst].mean()) / (cur["d_" + worst].std() + 1e-12)
    contrib = (zx * zy) * np.sign(cs[worst]["pearson"])
    out_i = contrib.sample(n=min(5, len(contrib)), weights=(contrib.clip(lower=0) + 1e-9), random_state=it).index[0]
    spare = list(pool_by_cell[OK.at[out_i, "cell"]] - sel)
    if not spare: no_imp += 1; continue
    in_i = spare[int(rs.integers(len(spare)))]
    trial = (sel - {out_i}) | {in_i}
    o, c2 = objective(OK.loc[list(trial)])
    if o < best_obj: sel, best_obj, cs, no_imp = trial, o, c2, 0
    else: no_imp += 1
MAIN = OK.loc[sorted(sel)].copy()
after_obj, after_cs = objective(MAIN)
print("before", {p: round(v["pearson"], 3) for p, v in before_cs.items()}, "| after", {p: round(v["pearson"], 3) for p, v in after_cs.items()},
      "| swaps", it)
counts = MAIN.groupby(["dstress", "cond"]).size().unstack(fill_value=0)
print(counts)

before {'nu': 0.07, 'eld': 0.062, 'xr': 0.084} | after {'nu': 0.041, 'eld': 0.056, 'xr': 0.071} | swaps 9798
cond     conflict_all  conflict_eld  conflict_nu  conflict_xr  congruent  \
dstress                                                                    
0.00               60            60           60           60         58   
0.03               60            60           60           60         59   
0.06               60            60           60           60         60   
0.10               60            60           60           60         60   
0.15               53            60           59           58         60   
0.20               13            24           13           13         47   

cond     neutral  
dstress           
0.00          60  
0.03          60  
0.06          60  
0.10          60  
0.15          60  
0.20          29  


## 4. Reference sets for Fig 3b: naive natural pairs and Mooney-style pairs

In [6]:
def naive_pairs(P):
    rows = []
    for (fam, n, g), grp in P.groupby(["family", "n", "graph"]):
        recs = grp.to_dict("records")
        for a, b in itertools.combinations(recs, 2):
            lo, hi = (a, b) if a["ksm"] <= b["ksm"] else (b, a)
            rows.append({"family": fam, "n": n, "graph": g, "A_id": lo["drawing_id"], "B_id": hi["drawing_id"],
                         **{"d_" + p: hi[p] - lo[p] for p in ["ksm"] + PROX}})
    return pd.DataFrame(rows)
NAIVE = naive_pairs(POOL[POOL.graph.isin(EVAL_GRAPHS)])
MO = MO02[MO02.apply(lambda r: (r.family, int(r.n)) in FAMILY_SETS, axis=1)].copy()
MO["n"] = MO.n.astype(int); MO["graph"] = MO.graph.astype(int)
MO["drawing_id"] = MO.apply(lambda r: f"M_{r.family}{r.n}_g{r.graph}_s{int(round(r.stress_target*100))}", axis=1)
MOP = naive_pairs(MO.assign(graph=MO.graph))
REF = {"mooney_style_pairs": corr_stats(MOP), "naive_natural_pairs": corr_stats(NAIVE),
       "pdcp_before_stage2": before_cs, "pdcp_after_stage2": after_cs}
for k, v in REF.items(): print(f"{k:24s}", {p: round(x["pearson"], 3) for p, x in v.items()})

# E1b: Mooney-style pairs balanced over dstress
MOP["dbin"] = (MOP.d_ksm / 0.05).round() * 0.05
e1b = []
per = max(1, N_E1B // MOP.dbin.nunique())
for d, g in MOP[MOP.dbin >= 0.05].groupby("dbin"):
    e1b.append(g.sample(n=min(per, len(g)), random_state=SEED))
E1B = pd.concat(e1b).head(N_E1B)
print(len(E1B), "E1b pairs; dstress bins", E1B.dbin.value_counts().sort_index().to_dict())

mooney_style_pairs       {'nu': 0.801, 'eld': 0.753, 'xr': 0.205}
naive_natural_pairs      {'nu': 0.511, 'eld': 0.386, 'xr': 0.739}
pdcp_before_stage2       {'nu': 0.07, 'eld': 0.062, 'xr': 0.084}
pdcp_after_stage2        {'nu': 0.041, 'eld': 0.056, 'xr': 0.071}
289 E1b pairs; dstress bins {0.05: 37, 0.1: 37, 0.15000000000000002: 37, 0.2: 37, 0.25: 37, 0.30000000000000004: 37, 0.35000000000000003: 37, 0.4: 30}


## 5. Subset (400) + counterfactual edits A′ (E2) + few-shot pool

In [7]:
MAIN["pair_id"] = [f"P{i:05d}" for i in range(len(MAIN))]
rs = np.random.default_rng(7)
MAIN["_r"] = rs.random(len(MAIN))
MAIN["subset400"] = False
strat = MAIN.sort_values("_r").groupby(["family", "n", "dstress", "cond"]).head(1)
extra = MAIN.drop(strat.index).sort_values("_r").head(max(0, N_SUBSET - len(strat)))
sub_idx = list(strat.index[:N_SUBSET]) + list(extra.index)
MAIN.loc[sub_idx[:N_SUBSET], "subset400"] = True
print("subset", MAIN.subset400.sum())

def gen_cf(job):
    import sys; sys.path.insert(0, OUTPUT_DIR)
    import numpy as np, torch, pdcp_utils as U; torch.set_num_threads(1)
    gc = U.GraphCache(U.make_graph(job["family"], job["n"], job["graph_seed"]))
    tg = {k: job["A_" + k] for k in ["ksm"] + PROX}; tg[job["proxy"]] += job["delta"]
    r = U.pdsg_optimise(gc, tg, X0=np.array(job["A_coords"]).reshape(-1, 2), tol=TOL_A, max_steps=1200, max_restarts=1, seed=job["seed"])
    m = U.all_metrics(r.X, gc, with_realism=True)
    return {"pair_id": job["pair_id"], "proxy": job["proxy"], "delta": job["delta"], "ok": bool(r.converged and m["realistic"]),
            **{k: m[k] for k in ["ksm", "nu", "eld", "xr", "aspect", "nn_max", "nn_min", "dense"]},
            "coords": r.X.ravel().round(6).tolist()}
coordsA = CAND.set_index("cand_id").coords
cfjobs = []
for r in MAIN[MAIN.subset400].itertuples():
    for p in PROX:
        for d in CF_DELTAS[p]:
            cfjobs.append({"pair_id": r.pair_id, "family": r.family, "n": int(r.n), "graph_seed": int(r.graph_seed),
                           "proxy": p, "delta": float(d), "A_coords": coordsA[r.cand_id],
                           **{"A_" + k: float(getattr(r, k)) for k in ["ksm"] + PROX}, "seed": int(rs.integers(1e8))})
print(len(cfjobs), "counterfactual jobs")
CF = pd.DataFrame(Parallel(n_jobs=N_JOBS, verbose=2)(delayed(gen_cf)(j) for j in cfjobs)) if elapsed_h() < MAX_HOURS else pd.DataFrame()
print("CF ok rate", round(CF.ok.mean(), 3) if len(CF) else None, f"| {elapsed_h()*60:.0f} min")
if len(CF): print(CF.groupby(["proxy", "delta"]).ok.mean().round(2).to_dict())

FS = CAND[CAND.ok & (CAND.role == "fewshot")].copy()
FS = FS.groupby(["family", "n"]).head(max(1, N_FEWSHOT // max(1, len(FAMILY_SETS)))).head(N_FEWSHOT)
for p in ["ksm"] + PROX: FS["d_" + p] = FS.B_id.map(Bm[p]) - FS[p]
print(len(FS), "few-shot pairs")

subset 400
3200 counterfactual jobs


[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  33 tasks      | elapsed:   32.0s
[Parallel(n_jobs=4)]: Done 154 tasks      | elapsed:  1.6min
[Parallel(n_jobs=4)]: Done 357 tasks      | elapsed:  4.7min
[Parallel(n_jobs=4)]: Done 640 tasks      | elapsed:  9.2min
[Parallel(n_jobs=4)]: Done 1005 tasks      | elapsed: 12.6min
[Parallel(n_jobs=4)]: Done 1450 tasks      | elapsed: 18.9min
[Parallel(n_jobs=4)]: Done 1977 tasks      | elapsed: 27.7min
[Parallel(n_jobs=4)]: Done 2584 tasks      | elapsed: 40.3min


CF ok rate 0.887 | 238 min
{('eld', 0.05): 0.96, ('eld', 0.1): 0.97, ('eld', 0.15): 0.94, ('nu', 0.05): 0.88, ('nu', 0.1): 0.54, ('xr', 0.02): 0.97, ('xr', 0.04): 0.94, ('xr', 0.06): 0.89}
30 few-shot pairs


[Parallel(n_jobs=4)]: Done 3200 out of 3200 | elapsed: 48.2min finished


## 6. Assemble drawings.parquet / pairs.parquet and render PNGs

In [8]:
MET = ["ksm", "nu", "eld", "xr", "aspect", "nn_max", "nn_min", "dense"]
def answer(d_ksm, thr=CFG["same_threshold"]):
    return "same" if abs(d_ksm) < thr else ("A" if d_ksm > 0 else "B")   # which drawing has LOWER stress
D, Pp = [], []
usedB = set(MAIN.B_id) | set(FS.B_id)
for r in POOL[POOL.drawing_id.isin(usedB)].itertuples():
    D.append({"drawing_id": r.drawing_id, "role": "B", "family": r.family, "n": r.n, "graph": r.graph, "graph_seed": r.graph_seed,
              **{k: getattr(r, k) for k in MET}, "coords": r.coords})
for df, role, prefix in [(MAIN, "A", "A_"), (FS, "A_fewshot", "F_")]:
    for r in df.itertuples():
        did = prefix + (r.pair_id if role == "A" else r.cand_id)
        D.append({"drawing_id": did, "role": role, "family": r.family, "n": int(r.n), "graph": int(r.graph), "graph_seed": int(r.graph_seed),
                  **{k: getattr(r, k) for k in MET}, "coords": coordsA[r.cand_id]})
        Pp.append({"pair_id": r.pair_id if role == "A" else "F" + r.cand_id[1:], "set": "main" if role == "A" else "fewshot",
                   "drawing_A": did, "drawing_B": r.B_id, "family": r.family, "n": int(r.n), "graph": int(r.graph),
                   "dstress_design": r.dstress, "cond": r.cond, **{"d_" + p: getattr(r, "d_" + p) for p in ["ksm"] + PROX},
                   "correct": answer(r.d_ksm), "subset400": bool(getattr(r, "subset400", False)),
                   "cf_proxy": None, "cf_delta": None, "base_pair": None})
mainmap = MAIN.set_index("pair_id")
for r in CF[CF.ok].itertuples() if len(CF) else []:
    base = mainmap.loc[r.pair_id]; did = f"CF_{r.pair_id}_{r.proxy}_{int(round(r.delta*100)):02d}"
    D.append({"drawing_id": did, "role": "CF", "family": base.family, "n": int(base.n), "graph": int(base.graph),
              "graph_seed": int(base.graph_seed), **{k: getattr(r, k) for k in MET}, "coords": r.coords})
    dk = Bm.loc[base.B_id, "ksm"] - r.ksm
    Pp.append({"pair_id": f"{r.pair_id}_{r.proxy}{int(round(r.delta*100)):02d}", "set": "counterfactual", "drawing_A": did,
               "drawing_B": base.B_id, "family": base.family, "n": int(base.n), "graph": int(base.graph),
               "dstress_design": base.dstress, "cond": base.cond, "d_ksm": dk,
               **{"d_" + p: Bm.loc[base.B_id, p] - getattr(r, p) for p in PROX}, "correct": answer(dk), "subset400": True,
               "cf_proxy": r.proxy, "cf_delta": r.delta, "base_pair": r.pair_id})
Mi = MO.set_index("drawing_id")
for i, r in enumerate(E1B.itertuples()):
    for did in (r.A_id, r.B_id):
        if not any(x["drawing_id"] == did for x in D[-2:]):
            m = Mi.loc[did]
            D.append({"drawing_id": did, "role": "M", "family": m.family, "n": int(m.n), "graph": int(m.graph), "graph_seed": int(m.graph_seed),
                      **{k: m[k] for k in MET}, "coords": list(m.coords)})
    Pp.append({"pair_id": f"E{i:04d}", "set": "e1b_mooney", "drawing_A": r.A_id, "drawing_B": r.B_id, "family": r.family, "n": int(r.n),
               "graph": int(r.graph), "dstress_design": float(r.dbin), "cond": "mooney", **{"d_" + p: getattr(r, "d_" + p) for p in ["ksm"] + PROX},
               "correct": answer(r.d_ksm), "subset400": False, "cf_proxy": None, "cf_delta": None, "base_pair": None})
DR = pd.DataFrame(D).drop_duplicates("drawing_id").reset_index(drop=True)
PR = pd.DataFrame(Pp)
rs2 = np.random.default_rng(11)
PR["default_order"] = np.where(rs2.random(len(PR)) < 0.5, "AB", "BA")      # which drawing is shown as Image 1 by default
DR["png"] = DR.drawing_id.map(lambda d: f"images/{RES}/{d}.png")
os.makedirs(f"{OUTPUT_DIR}/images/{RES}", exist_ok=True)

def render_one(rec):
    import sys; sys.path.insert(0, OUTPUT_DIR)
    import numpy as np, pdcp_utils as U
    gc = U.GraphCache(U.make_graph(rec["family"], rec["n"], rec["graph_seed"]))
    U.render_png(gc, np.array(rec["coords"]).reshape(-1, 2), f"{OUTPUT_DIR}/{rec['png']}", res=RES)
    return 1
Parallel(n_jobs=N_JOBS, verbose=1)(delayed(render_one)(r) for r in DR.to_dict("records"))
DR.to_parquet(f"{OUTPUT_DIR}/drawings.parquet", index=False)
PR.to_parquet(f"{OUTPUT_DIR}/pairs.parquet", index=False)
print(len(DR), "drawings |", PR.set.value_counts().to_dict(), "|", PR[PR.set == "main"].correct.value_counts().to_dict(), f"| {elapsed_h()*60:.0f} min")

[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    6.7s
[Parallel(n_jobs=4)]: Done 192 tasks      | elapsed:   30.0s
[Parallel(n_jobs=4)]: Done 442 tasks      | elapsed:  1.1min
[Parallel(n_jobs=4)]: Done 792 tasks      | elapsed:  2.0min
[Parallel(n_jobs=4)]: Done 1242 tasks      | elapsed:  3.1min
[Parallel(n_jobs=4)]: Done 1792 tasks      | elapsed:  4.5min
[Parallel(n_jobs=4)]: Done 2442 tasks      | elapsed:  6.0min
[Parallel(n_jobs=4)]: Done 3192 tasks      | elapsed:  7.8min
[Parallel(n_jobs=4)]: Done 4042 tasks      | elapsed:  9.8min
[Parallel(n_jobs=4)]: Done 4992 tasks      | elapsed: 12.0min


5315 drawings | {'counterfactual': 2839, 'main': 1926, 'e1b_mooney': 289, 'fewshot': 30} | {'A': 1568, 'same': 358} | 251 min


[Parallel(n_jobs=4)]: Done 5315 out of 5315 | elapsed: 12.8min finished


## 7. Figures (F2 gallery, F3b correlation matrices, cell counts) and checks

In [9]:
# F3b: corr(dKSM, dProxy) across stimulus sets
sets = ["mooney_style_pairs", "naive_natural_pairs", "pdcp_before_stage2", "pdcp_after_stage2"]
titles = ["Mooney-style\n(stress-only, 0.40-0.80)", "Naive natural pairs\n(0.30-0.50)", "PDCP before Stage 2", "PDCP after Stage 2 (final)"]
M = np.array([[REF[s][p]["pearson"] for p in PROX] for s in sets])
fig, ax = plt.subplots(figsize=(6.5, 4.2))
im = ax.imshow(M, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(3)); ax.set_xticklabels(["Δ node\nuniformity", "Δ edge-length\ndeviation", "Δ crossings"], fontsize=9)
ax.set_yticks(range(4)); ax.set_yticklabels(titles, fontsize=8)
for i in range(4):
    for j in range(3): ax.text(j, i, f"{M[i,j]:+.2f}", ha="center", va="center", fontsize=10, color="white" if abs(M[i,j]) > .5 else "black")
ax.set_title("Pearson r(Δstress, Δproxy) across pairs"); fig.colorbar(im, fraction=0.03)
plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/fig_F3b_decorrelation.png", dpi=150); plt.show()

# cell counts heatmap
cc = PR[PR.set == "main"].groupby(["dstress_design", "cond"]).size().unstack(fill_value=0)[list(CONDS)]
fig, ax = plt.subplots(figsize=(8, 3.4)); ax.imshow(cc.values, cmap="Greens")
ax.set_xticks(range(len(CONDS))); ax.set_xticklabels(list(CONDS), rotation=25); ax.set_yticks(range(len(cc))); ax.set_yticklabels(cc.index)
for i in range(cc.shape[0]):
    for j in range(cc.shape[1]): ax.text(j, i, cc.values[i, j], ha="center", va="center")
ax.set_ylabel("Δstress"); ax.set_title("Main pairs per Δstress × condition (all families)")
plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/fig_cell_counts.png", dpi=130); plt.show()

# F2 gallery: one family, Δ ∈ {0, 0.10} × 6 conditions, showing A | B
from PIL import Image
fam0 = PR[PR.set == "main"].groupby(["family", "n"]).size().idxmax()
rows_g = []
for d in [0.0, 0.10]:
    for cond in CONDS:
        q = PR[(PR.set == "main") & (PR.family == fam0[0]) & (PR.n == fam0[1]) & (PR.dstress_design == d) & (PR.cond == cond)]
        if len(q): rows_g.append(q.iloc[0])
fig, ax = plt.subplots(len(rows_g), 2, figsize=(6, 3 * len(rows_g)))
Di = DR.set_index("drawing_id")
for i, r in enumerate(rows_g):
    for j, did in enumerate([r.drawing_A, r.drawing_B]):
        a = ax[i, j]; a.imshow(Image.open(f"{OUTPUT_DIR}/{Di.loc[did, 'png']}")); a.axis("off")
        m = Di.loc[did]
        a.set_title(f"{'A' if j == 0 else 'B'}  KSM {m.ksm:.3f} NU {m.nu:.2f} ELD {m.eld:.2f} XR {m.xr:.2f}", fontsize=7)
    ax[i, 0].text(-0.08, 0.5, f"Δ={r.dstress_design:.2f}\n{r.cond}", transform=ax[i, 0].transAxes, ha="right", va="center", fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/fig_F2_gallery.png", dpi=100); plt.show()

# checks
main = PR[PR.set == "main"]
chk = {"n_main": int(len(main)), "n_counterfactual": int((PR.set == "counterfactual").sum()), "n_e1b": int((PR.set == "e1b_mooney").sum()),
       "n_fewshot": int((PR.set == "fewshot").sum()), "n_drawings": int(len(DR)),
       "pngs_present": int(sum(os.path.exists(f"{OUTPUT_DIR}/{p}") for p in DR.png)),
       "same_pairs_have_tiny_dksm": bool((main[main.dstress_design == 0].d_ksm.abs() < CFG["same_threshold"]).mean() > 0.95),
       "fewshot_graphs_disjoint": bool(set(PR[PR.set == "fewshot"].graph) .isdisjoint(set(main.graph))),
       "corr_after": {p: round(v["pearson"], 4) for p, v in after_cs.items()},
       "corr_after_spearman": {p: round(v["spearman"], 4) for p, v in after_cs.items()}}
print(json.dumps(chk, indent=1))

{
 "n_main": 1926,
 "n_counterfactual": 2839,
 "n_e1b": 289,
 "n_fewshot": 30,
 "n_drawings": 5315,
 "pngs_present": 5315,
 "same_pairs_have_tiny_dksm": true,
 "fewshot_graphs_disjoint": true,
 "corr_after": {
  "nu": 0.0406,
  "eld": 0.056,
  "xr": 0.0706
 },
 "corr_after_spearman": {
  "nu": -0.0,
  "eld": 0.003,
  "xr": 0.0001
 }
}


## 8. Report and manifest

In [10]:
REPORT = {"utils_version": U.__version__, "runtime_min": round(elapsed_h() * 60, 1), "fast": FAST,
          "design": {"stress_levels": STRESS_LEVELS, "dstress": DSTRESS, "conditions": CONDS, "margins": MARGIN, "tol_A": TOL_A,
                     "per_cell": PER_CELL, "candidates_per_cell": CANDIDATES, "cf_deltas": CF_DELTAS, "corr_limit": CORR_LIMIT},
          "stage1_ok_rate": float(CAND.ok.mean()), "stage1_ok_by_dstress": CAND.groupby("dstress").ok.mean().round(3).to_dict(),
          "stage1_ok_by_cond": CAND.groupby("cond").ok.mean().round(3).to_dict(),
          "stage1_ok_by_family": (CAND.family + CAND.n.astype(str)).to_frame("f").assign(ok=CAND.ok).groupby("f").ok.mean().round(3).to_dict(),
          "stage2_swaps": it, "correlations": REF, "cf_ok_rate": float(CF.ok.mean()) if len(CF) else None,
          "cf_ok_by": {f"{k[0]}+{k[1]}": v for k, v in CF.groupby(["proxy", "delta"]).ok.mean().round(3).items()} if len(CF) else {},
          "checks": chk}
json.dump(REPORT, open(f"{OUTPUT_DIR}/stage2_report.json", "w"), indent=2, default=str)
shutil.rmtree(f"{OUTPUT_DIR}/__pycache__", ignore_errors=True)
desc = {"pdcp_utils.py": "shared module v0.1.2 (int seeds) - use from now on",
        "drawings.parquet": "one row per drawing: coords, exact metrics, realism, png path",
        "pairs.parquet": "one row per pair: sets main/counterfactual/e1b_mooney/fewshot, d_* = B minus A, correct answer",
        "candidates.parquet": "every Stage-1 attempt (feasibility record, no coords)",
        "stage2_report.json": "design, success rates, correlations before/after, checks",
        "fig_F3b_decorrelation.png": "paper Fig 3b", "fig_F2_gallery.png": "paper Fig 2 (stimulus gallery)",
        "fig_cell_counts.png": "pairs per cell"}
lines = [f"N03 output · utils {U.__version__} · {REPORT['runtime_min']} min · main {chk['n_main']} · CF {chk['n_counterfactual']} · "
         f"E1b {chk['n_e1b']} · fewshot {chk['n_fewshot']} · drawings {chk['n_drawings']} (png {chk['pngs_present']})",
         f"corr after Stage 2 (pearson): {chk['corr_after']}", ""]
for f in sorted(os.listdir(OUTPUT_DIR)):
    p = os.path.join(OUTPUT_DIR, f)
    if os.path.isfile(p): lines.append(f"{f:30s} {os.path.getsize(p)/1024:10.1f} KB   {desc.get(f, '')}")
lines.append(f"images/{RES}/  {len(os.listdir(f'{OUTPUT_DIR}/images/{RES}'))} PNG files")
open(f"{OUTPUT_DIR}/MANIFEST.txt", "w").write("\n".join(lines)); print("\n".join(lines))

N03 output · utils 0.1.2 · 251.0 min · main 1926 · CF 2839 · E1b 289 · fewshot 30 · drawings 5315 (png 5315)
corr after Stage 2 (pearson): {'nu': 0.0406, 'eld': 0.056, 'xr': 0.0706}

candidates.parquet                  320.6 KB   every Stage-1 attempt (feasibility record, no coords)
drawings.parquet                   5739.4 KB   one row per drawing: coords, exact metrics, realism, png path
fig_F2_gallery.png                  888.7 KB   paper Fig 2 (stimulus gallery)
fig_F3b_decorrelation.png            66.7 KB   paper Fig 3b
fig_cell_counts.png                  37.6 KB   pairs per cell
pairs.parquet                       250.0 KB   one row per pair: sets main/counterfactual/e1b_mooney/fewshot, d_* = B minus A, correct answer
pdcp_utils.py                        21.9 KB   shared module v0.1.2 (int seeds) - use from now on
stage2_report.json                    3.8 KB   design, success rates, correlations before/after, checks
images/768/  5315 PNG files
